# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
The ranked queue turns the model score into human-readable actions. Higher-scored items should be reviewed first. The main actions are Refresh, Improve, Monitor, or No immediate action. Reason codes explain why an item received its priority, such as declining visibility, freshness risk, position opportunity, content-depth gap, low CTR, or low engagement. The ranking is directional decision support and requires human review before action.

In [7]:
import pandas as pd
import numpy as np

# Load the model output created from the ML task
url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Create a simple directional review score from observable signals
# These are only used to create a transparent review queue.

df["visibility_score"] = (
    pd.to_numeric(df["impressions_90d"], errors="coerce")
    .fillna(0)
    .rank(pct=True)
)

df["freshness_score"] = (
    pd.to_numeric(df["days_since_last_update"], errors="coerce")
    .fillna(0)
    .rank(pct=True)
)

df["position_score"] = (
    pd.to_numeric(df["avg_position"], errors="coerce")
    .fillna(0)
    .rank(pct=True)
)

df["engagement_score"] = (
    pd.to_numeric(df["engagement_rate"], errors="coerce")
    .fillna(0)
    .rank(pct=True)
)

# Higher score = higher review priority
df["action_score"] = (
    0.30 * df["visibility_score"] +
    0.30 * df["freshness_score"] +
    0.20 * df["position_score"] +
    0.20 * df["engagement_score"]
)

# Rank the queue
action_df = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

action_df["action_rank"] = np.arange(
    1,
    len(action_df) + 1
)

def assign_action(score):
    if score >= 0.70:
        return "Refresh"
    elif score >= 0.50:
        return "Improve"
    elif score >= 0.30:
        return "Monitor"
    else:
        return "No immediate action"

action_df["action"] = action_df["action_score"].apply(assign_action)

action_df["reason_code"] = np.select(
    [
        action_df["action_score"] >= 0.70,
        action_df["action_score"] >= 0.50,
        action_df["action_score"] >= 0.30
    ],
    [
        "HIGH_REVIEW_PRIORITY",
        "MEDIUM_REVIEW_PRIORITY",
        "LOW_REVIEW_PRIORITY"
    ],
    default="LOW_PRIORITY"
)

action_df["confidence_note"] = (
    "Directional score; human review required."
)

print("Ranked action queue created.")
print("Rows:", len(action_df))

display(
    action_df[
        [
            "action_rank",
            "content_id",
            "action_score",
            "action",
            "reason_code",
            "confidence_note"
        ]
    ].head(20)
)

Ranked action queue created.
Rows: 30000


,action_rank,content_id,action_score,action,reason_code,confidence_note
0,1,content_62abc4bd66be,0.934315,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
1,2,content_ac1d924c6a70,0.932053,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
2,3,content_fb66dd8f4629,0.928970,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
3,4,content_47b8b12d581e,0.921183,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
4,5,content_7368877ea310,0.918170,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
5,6,content_1f3b8e699416,0.913797,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
6,7,content_124763d39ca5,0.909637,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
7,8,content_90bb37b53856,0.907487,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
8,9,content_46b5a482fa35,0.907303,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.
9,10,content_0ff10b5a4bae,0.905997,Refresh,HIGH_REVIEW_PRIORITY,Directional score; human review required.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
The intended users are content and SEO reviewers who need to prioritize a limited review queue. The output can help them decide which content items deserve earlier investigation based on observed signals and model scores. It should not automatically publish, delete, merge, or rewrite content. The recommendations are only valid for data and conditions similar to the evaluated dataset and should be treated as directional decision support rather than guaranteed predictions.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use: human-assisted content review prioritization")
print("Automatic publishing: not allowed")
print("Automatic deletion or pruning: not allowed")
print("Interpretation: directional decision support")
print("Requires human review: yes")

Intended use: human-assisted content review prioritization
Automatic publishing: not allowed
Automatic deletion or pruning: not allowed
Interpretation: directional decision support
Requires human review: yes


## 3. Human review + the no-go list
A human reviewer must check the content context, recent changes, traffic and search trends, seasonality, and whether the recommendation makes sense before taking action. The system must not automatically publish, delete, merge, or substantially rewrite content. It also must not be used to claim causality or to predict Google's ranking algorithm.
*What a person must check before acting. What should never be automated.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go_actions = [
    "automatic publishing",
    "automatic deletion",
    "automatic content merging",
    "automatic substantial rewriting",
    "claiming causal Google ranking effects"
]

human_checks = [
    "content context",
    "recent changes",
    "traffic and search trends",
    "seasonality",
    "data quality",
    "recommendation reason"
]

print("Human checks required:")
for item in human_checks:
    print("-", item)

print("\nNo-go actions:")
for item in no_go_actions:
    print("-", item)

Human checks required:
- content context
- recent changes
- traffic and search trends
- seasonality
- data quality
- recommendation reason

No-go actions:
- automatic publishing
- automatic deletion
- automatic content merging
- automatic substantial rewriting
- claiming causal Google ranking effects


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
The recommendations should be reviewed when the data distribution changes, important input fields become unavailable, the share of declining items changes substantially, or review outcomes show that the ranked recommendations are becoming less useful. A retraining or recalibration review should also be triggered when the model's validation performance drops materially compared with the original evaluation.

In [10]:
# Monitoring checks

declining_rate = df["trend_direction"].astype(str).str.lower().eq("down").mean()

print("Current declining rate:", round(declining_rate, 3))
print("Rows available:", len(df))
print("Feature columns available:", len(feature_cols) if "feature_cols" in globals() else "N/A")

print("\nRetrain/review triggers:")
print("- Material change in input-data distribution")
print("- Important feature becomes unavailable")
print("- Large change in declining-content rate")
print("- Validation performance drops materially")
print("- Human reviewers report declining recommendation quality")

Current declining rate: 0.542
Rows available: 30000
Feature columns available: N/A

Retrain/review triggers:
- Material change in input-data distribution
- Important feature becomes unavailable
- Large change in declining-content rate
- Validation performance drops materially
- Human reviewers report declining recommendation quality


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
I will export the ranked action queue and summary statistics so the final research paper can reuse the same results. The exported files are decision-support artifacts and do not contain private client names, URLs, credentials, or raw private queries.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export ranked action queue
queue_columns = [
    "action_rank",
    "content_id",
    "action_score",
    "action",
    "reason_code",
    "confidence_note"
]

queue_path = output_dir / "content_action_playbook.csv"

action_df[queue_columns].to_csv(
    queue_path,
    index=False
)

# Export summary
summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "refresh_count",
        "improve_count",
        "monitor_count",
        "no_immediate_action_count"
    ],
    "value": [
        len(action_df),
        (action_df["action"] == "Refresh").sum(),
        (action_df["action"] == "Improve").sum(),
        (action_df["action"] == "Monitor").sum(),
        (action_df["action"] == "No immediate action").sum()
    ]
})

summary_path = output_dir / "content_action_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print("Queue saved to:", queue_path)
print("Summary saved to:", summary_path)

display(summary)

Queue saved to: work/outputs/content_action_playbook.csv
Summary saved to: work/outputs/content_action_summary.csv


,metric,value
0,total_rows,30000
1,refresh_count,3975
2,improve_count,11900
3,monitor_count,9823
4,no_immediate_action_count,4302


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.